# Evaluación de Modelos de Regresión Lineal

Esta guía interactiva examina los **fundamentos de la evaluación de regresión lineal**: cómo se generan datos sintéticos, cómo se calculan métricas de desempeño y cómo se interpretan visualizaciones de error.

El análisis recorre tres escenarios progresivamente más difíciles — datos ideales, datos con ruido y datos con outliers — con el fin de evidenciar la sensibilidad del modelo frente a imperfecciones del mundo real.

---

## Prerrequisitos

Se asume familiaridad con:
- Álgebra lineal básica (ecuación de la recta `y = mx + b`)
- Python y NumPy a nivel introductorio
- Conceptos de media y varianza

## Objetivos de aprendizaje

Al finalizar esta guía, el lector será capaz de:
1. Generar datos sintéticos con ruido controlado
2. Calcular e interpretar MSE, RMSE, MAE y R²
3. Detectar outliers visualmente mediante gráficos de residuos
4. Comparar el desempeño de un modelo en distintos escenarios

---

## Índice

1. Importación de librerías
2. Configuración visual
3. Generación de datos base
4. Simulación de escenarios reales
5. Métricas de evaluación
6. Función de evaluación reutilizable
7. Comparación entre escenarios
8. Visualización: datos reales vs. predicciones
9. Análisis de residuos
10. Comparación de métricas de error
11. Conclusiones

## 1. Importación de librerías

La regresión lineal requiere un conjunto de herramientas especializadas. A continuación se describen las librerías (lib.) empleadas y su rol dentro del análisis:

- **NumPy**: operaciones numéricas y generación de datos aleatorios con semilla controlada
- **Matplotlib**: creación de visualizaciones estáticas
- **Seaborn**: estilos mejorados aplicados sobre Matplotlib
- **scikit-learn** (`sklearn`): implementación del modelo de regresión lineal y cálculo de métricas de evaluación (eval.)
- **Pandas**: organización de resultados en estructuras tabulares (DataFrames)

In [ ]:
import numpy as np                                          # Operaciones numéricas
import matplotlib.pyplot as plt                             # Visualización
import seaborn as sns                                       # Estilos mejorados para gráficos
import pandas as pd                                         # Tablas de datos (DataFrames)
from sklearn.linear_model import LinearRegression           # Modelo de regresión lineal
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error  # Métricas de eval.

## 2. Configuración visual

Antes de generar datos, se establece un tema oscuro para los gráficos. Esta configuración mejora la legibilidad y asegura consistencia visual a lo largo de toda la guía.

Los parámetros `rcParams` de Matplotlib permiten controlar globalmente colores de fondo, ejes, etiquetas y marcas de escala.

In [ ]:
# Aplicar tema oscuro
plt.style.use('dark_background')
sns.set_theme(style="darkgrid")

# Personalizar colores y estilos
plt.rcParams['figure.facecolor'] = '#1a1a1a'
plt.rcParams['axes.facecolor'] = '#2b2b2b'
plt.rcParams['axes.edgecolor'] = '#444444'
plt.rcParams['text.color'] = '#ffffff'
plt.rcParams['axes.labelcolor'] = '#ffffff'
plt.rcParams['xtick.color'] = '#ffffff'
plt.rcParams['ytick.color'] = '#ffffff'

## 3. Generación de datos base

El análisis parte de datos sintéticos que siguen una relación lineal perfecta: **y = 2x + 5**.

Trabajar con datos cuya verdad subyacente se conoce permite evaluar con precisión qué tan bien el modelo recupera esa relación. La función `np.random.seed(42)` fija el generador aleatorio, garantizando **reproducibilidad**: cualquier ejecución producirá exactamente los mismos números.

- `X`: variable independiente (feat.) con 200 puntos en el rango [0, 50]
- `y`: variable dependiente, calculada directamente desde la ecuación

In [ ]:
# Fijar la semilla para reproducibilidad
np.random.seed(42)

# Crear datos X (variable independiente) de 0 a 50 con 200 puntos
X = np.linspace(0, 50, 200)

# Crear datos Y que siguen la relación lineal: y = 2x + 5
y = 2 * X + 5

## 4. Simulación de escenarios reales

En la práctica, los datos nunca son perfectos. Por ello, se construyen tres variantes del vector `y` para simular condiciones del mundo real:

1. **Ruido pequeño** (`y_pred`): ruido gaussiano con desviación estándar σ = 2.5 — simula mediciones ligeramente imprecisas, e.g., errores de instrumento
2. **Ruido grande** (`y_pred_noise`): ruido gaussiano con σ = 10 — simula mediciones muy imprecisas
3. **Outliers** (`y_pred_outliers`): copia del caso con ruido pequeño, pero con cuatro errores extremos (+/− 40 unidades) inyectados manualmente en posiciones específicas

Estos tres escenarios permiten comparar cómo el modelo responde ante niveles crecientes de imperfección.

In [ ]:
# Caso 1: Datos con ruido pequeño (datos realistas)
y_pred = y + np.random.normal(0, 2.5, 200)

# Caso 2: Datos con ruido grande (mediciones muy imprecisas)
y_pred_noise = y + np.random.normal(0, 10, 200)

# Caso 3: Datos con outliers (valores anómalos)
y_pred_outliers = y_pred.copy()
y_pred_outliers[25] += 40    # Error grande en posición 25
y_pred_outliers[50] -= 40    # Error grande en posición 50
y_pred_outliers[100] += 40   # Error grande en posición 100
y_pred_outliers[150] += 40   # Error grande en posición 150

## 5. Métricas de evaluación

Para cuantificar el desempeño de un modelo, se emplean métricas de error. Cada una captura una dimensión distinta del comportamiento del modelo:

| Métrica | Fórmula | Interpretación |
|---------|---------|----------------|
| **MSE** (Error Cuadrático Medio) | `mean((y_real - y_pred)²)` | Penaliza fuertemente errores grandes; sensible a outliers |
| **RMSE** (Raíz del Error Cuadrático Medio) | `sqrt(MSE)` | Mismas unidades que `y`; más interpretable que MSE |
| **MAE** (Error Absoluto Medio) | `mean(|y_real - y_pred|)` | Robusto frente a outliers; interpretación directa |
| **R²** (Coeficiente de Determinación) | Proporción de varianza explicada | Rango [0, 1]; valores cercanos a 1 → modelo ajustado |

**Nb**: el MSE amplifica errores grandes al elevarlos al cuadrado. En consecuencia, el escenario con outliers producirá un MSE desproporcionadamente alto respecto al MAE.

La siguiente celda calcula estas métricas para el escenario de ruido pequeño como ejemplo inicial:

In [ ]:
# Ejemplo: Calculemos estas métricas para el primer caso (ruido pequeño)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)

print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R²: {r2:.4f}")

## 6. Función de evaluación reutilizable

El principio **DRY** (*Don't Repeat Yourself*) establece que el cálculo repetitivo debe encapsularse en una función. La función `model_evaluate` recibe el nombre del escenario, los valores reales y las predicciones, y retorna un diccionario con las cuatro métricas.

Este enfoque permite evaluar múltiples escenarios con una sola llamada por caso, eliminando duplicación de código y reduciendo la posibilidad de errores.

In [ ]:
def model_evaluate(name: str, true, pred):
    mse = mean_squared_error(true, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(true, pred)
    r2 = r2_score(true, pred)

    return {
        'model': name, 
        'mse': mse, 
        'rmse': rmse, 
        'mae': mae, 
        'r2': r2
    }

## 7. Comparación entre escenarios

`model_evaluate` se aplica a los tres casos definidos anteriormente. Los resultados se consolidan en un DataFrame de Pandas, que facilita la comparación tabular.

**Hipótesis a verificar:** el escenario *Ideal* debería mostrar el R² más alto y el menor error; el escenario *Con Outliers* debería presentar el MSE más elevado, mientras que el MAE permanece relativamente contenido — evidenciando la diferencia de sensibilidad entre ambas métricas.

In [ ]:
# Evaluamos el modelo en todos los escenarios
resultados = [
    model_evaluate('Ideal', y, y_pred),
    model_evaluate('Con Ruido', y, y_pred_noise),
    model_evaluate('Con Outliers', y, y_pred_outliers)
]

# Convertir a DataFrame para una mejor visualización
data = pd.DataFrame(resultados).set_index('model')
data

## 8. Visualización: datos reales vs. predicciones

El gráfico a continuación superpone los tres escenarios sobre la línea ideal, permitiendo apreciar visualmente el efecto del ruido y los outliers:

- **Línea gris**: relación lineal perfecta `y = 2x + 5` (ground truth)
- **Puntos cyan**: predicciones con ruido pequeño — dispersión moderada alrededor de la línea ideal
- **Puntos naranja**: predicciones con ruido grande — dispersión significativamente mayor
- **Puntos rojos**: predicciones con outliers — idénticos al caso cyan, salvo por cuatro valores extremos visibles

Esta visualización complementa la tabla de métricas: mientras los números cuantifican el error, el gráfico permite identificar **en qué regiones del espacio** se concentran las desviaciones.

In [ ]:
figure = plt.figure(figsize=(15, 8))

ax1 = figure.add_subplot()
ax1.plot(X, y, color='lightgray', linewidth=2, label='Datos Reales (Ideales)')
ax1.scatter(X, y_pred, color='cyan', alpha=0.5, s=30, label='Con Ruido Pequeño')
ax1.scatter(X, y_pred_noise, color='orange', alpha=0.5, s=30, label='Con Ruido Grande')
ax1.scatter(X, y_pred_outliers, color='red', alpha=0.7, s=50, label='Con Outliers')

ax1.set_xlabel('X (Variable Independiente)', fontsize=12)
ax1.set_ylabel('Y (Variable Dependiente)', fontsize=12)
ax1.set_title('Comparación de Datos Reales vs Predicciones', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Análisis de residuos

El **residuo** es la diferencia entre el valor real y la predicción: `error = y_real − y_pred`. Un modelo ideal produciría residuos distribuidos aleatoriamente alrededor de cero, sin patrones sistemáticos.

El gráfico de residuos (*stem plot*) permite:
- Identificar zonas donde el modelo falla consistentemente
- Detectar outliers como picos aislados de gran magnitud
- Verificar si los errores son homogéneos a lo largo del rango de `X`

**[!]** Los cuatro outliers inyectados en el escenario rojo aparecen como picos de ±40 unidades, claramente distinguibles del resto del patrón de error.

In [ ]:
figure = plt.figure(figsize=(15, 8))

ax2 = figure.add_subplot()

# Gráfico de residuos del caso con ruido
ax2.stem(X, y - y_pred, linefmt='cyan', markerfmt='co', label='Residuos (Ruido Pequeño)')

# Gráfico de residuos del caso con outliers (es más evidente)
ax2.stem(X, y - y_pred_outliers, linefmt='r-', markerfmt='rx', label='Residuos (Outliers)')

ax2.axhline(y=0, color='white', linestyle='--', alpha=0.5, linewidth=1)
ax2.set_xlabel('X', fontsize=12)
ax2.set_ylabel('Residuo (Error)', fontsize=12)
ax2.set_title('Análisis de Residuos: Donde el Modelo se Equivoca', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Comparación de métricas de error

El gráfico de barras compara MAE y RMSE entre los tres escenarios, facilitando la lectura cruzada de la tabla generada en la sección 7.

**Interpretación esperada:**
- El escenario *Con Ruido* presentará valores intermedios en ambas métricas
- El escenario *Con Outliers* mostrará un RMSE desproporcionadamente alto respecto al MAE — dado que el cuadrado amplifica los errores extremos
- La brecha entre MAE y RMSE es, por lo tanto, un indicador de la presencia de outliers en los datos

Este análisis visual complementa la tabla numérica y facilita la comparación rápida entre escenarios.

In [ ]:
figure = plt.figure(figsize=(15, 8))

ax3 = figure.add_subplot()

# Crear gráfico de barras para comparar MAE y RMSE
data[['mae', 'rmse']].plot(
    kind='bar', 
    ax=ax3,
    color=['#00ffff', '#ff00ff'],
    alpha=0.8,
    width=0.7
)

ax3.set_xlabel('Escenario', fontsize=12)
ax3.set_ylabel('Error', fontsize=12)
ax3.set_title('Comparación de Métricas de Error entre Escenarios', fontsize=14, fontweight='bold')
ax3.legend(['MAE (Error Absoluto Medio)', 'RMSE (Raíz del Error Cuadrático)'], fontsize=10)
ax3.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 11. Conclusiones

El análisis realizado a lo largo de esta guía permite establecer las siguientes conclusiones:

1. **La regresión lineal es interpretable pero sensible**: el modelo recupera con precisión la relación `y = 2x + 5` en condiciones ideales; sin embargo, su desempeño se degrada en presencia de ruido o valores atípicos.

2. **Los datos reales contienen imperfecciones**: en consecuencia, la evaluación de un modelo debe realizarse bajo múltiples escenarios antes de concluir que el modelo es apto para producción.

3. **MSE vs. MAE capturan aspectos distintos del error**: el RMSE amplifica errores grandes (∴ es más sensible a outliers), mientras que el MAE los trata linealmente. La brecha entre ambas métricas es un diagnóstico de la distribución del error.

4. **Los outliers distorsionan las métricas cuadráticas**: un solo valor extremo puede elevar el MSE de forma desproporcionada, lo que motiva la limpieza de datos antes del entrenamiento.

5. **La visualización de residuos es indispensable**: los números solos no revelan *dónde* falla el modelo; el gráfico de residuos localiza espacialmente los errores.

---

**Próximos pasos sugeridos:**
- Aplicar técnicas de detección y remoción de outliers (IQR, z-score)
- Explorar regresión polinomial para relaciones no lineales
- Introducir validación cruzada (*k-fold*) para una eval. más robusta del desempeño